# BMLL | Execution simulation and TCA: closing auction vs VWAP vs arrival + impact

Phase 2 of the simulated $10B S&P 500 fund:

1. **Order schema.** PM → Trader → Broker → Fill tables (`tca/schema.py`, documented in `docs/order_schema.md`).
2. **Synthetic order generator.** It splits each PM order into trader orders by strategy, and then into child orders by time bucket and by **MIC**, in proportion to each venue's share of volume. The venue shares come from BMLL per-listing VWAP and volume (`tca/order_generator.py`).
3. **One rebalance date.** The same PM orders are executed three ways: **closing auction (MOC)**, **VWAP**, and **arrival price + impact**.
4. **All rebalances.** Notional-weighted costs and the **tracking error** that execution adds against the close-rebalanced index.

**Data modes**
* `DATA_SOURCE='bmll'` must run inside BMLL Data Lab. Daily per-venue `VWAP` and `TradeVolume` (lit and dark), plus opening and closing auction prices, come from `bmll.time_series`.
* `DATA_SOURCE='synthetic'` runs anywhere. It is anchored on the blotter's **real** decision-day and execution-day CRSP closes, so only the intraday path, venue split and liquidity are simulated. Synthetic results show the mechanics; they are **not** findings.

**Cost sign:** positive bps means a cost to the fund. An index fund is benchmarked to the **rebalance-day close**, so *cost vs close* is the tracking difference created by execution.

## 0. Configuration

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_SOURCE = 'synthetic'          # 'synthetic' or 'bmll' (inside BMLL Data Lab)
# Blotter. After the Friday-auction re-run, point these at the new candidate:
#   outputs/daily_fund/orders.csv + fills.csv + daily_nav.csv
# Until then the supervisor handoff (Monday-decision / Tuesday-fill timing) is used.
HANDOFF = Path('../BMLL_TCA_Supervisor_Handoff')
BLOTTER_CSV = HANDOFF/'orderbook_tca_handoff_split_normalized.csv'   # orders already joined to fills
ORDERS_CSV, FILLS_CSV = None, None                                    # or two files to join
NAV_CSV = HANDOFF/'portfolio_nav.csv'
WEIGHTS_CSV = Path('data/stage2_inputs/weights.csv')                  # market caps + PIT tickers
REASONS = ['REBALANCE']            # core TCA sample; add 'INFLOW','OUTFLOW' for flow trades
FOCUS_TRADE_DATE = None            # None -> the rebalance with the largest traded notional
ANALYSIS_START, END_DATE = '2022-07-01', '2026-06-30'
SEED = 11

# Execution assumptions (literature defaults; calibrate to BMLL before quoting results)
IMPACT = dict(eta=0.6, auction_eta=0.25, cross_fraction=0.5)
GEN = dict(max_auction_participation=.25, max_participation=.10, is_participation=.15)
OUT = Path('outputs/tca'); OUT.mkdir(parents=True, exist_ok=True)

## 1. PM orders from the fund blotter

INITIAL and REBALANCE orders are benchmarked to `CLOSE`; flow orders are benchmarked to `ARRIVAL`. Quantities and decision prices use the split-adjusted execution basis when it is available.

In [ ]:
from tca.schema import pm_orders_from_blotter, join_orders_fills, validate_links, SCHEMAS
if ORDERS_CSV:
    blotter = join_orders_fills(pd.read_csv(ORDERS_CSV), pd.read_csv(FILLS_CSV))
else:
    blotter = pd.read_csv(BLOTTER_CSV)
blotter = blotter[blotter.reason.isin(REASONS)]
trade_col = 'execution_date' if 'execution_date' in blotter else 'expected_execution_date'
pm = pm_orders_from_blotter(blotter, trade_date_col=trade_col)
nav = pd.read_csv(NAV_CSV, parse_dates=['date'])
weights = pd.read_csv(WEIGHTS_CSV)
print(f"{len(pm):,} PM orders on {pm.trade_date.nunique()} trade dates; "
      f"${pm.notional_usd.sum()/1e9:,.1f}B notional")
print('Trade weekdays:', pm.groupby(pm.trade_date.dt.day_name()).size().to_dict())
display(pm.head())

### Order schema

| Table | One row per | Key fields |
|---|---|---|
| `pm_orders` | Portfolio decision | side, quantity, decision price, reason, benchmark (CLOSE/ARRIVAL), urgency |
| `trader_orders` | Way of working a PM order | strategy, window, participation cap, primary MIC; a spill leg if capped |
| `broker_orders` | Child routed to a venue | MIC, order type (MOC/LIMIT/MIDPOINT_PEG), TIF, time bucket, collar |
| `fills` | Execution | MIC, time, price, liquidity flag, fee, modelled impact |

`validate_links` checks foreign keys, that each child has the same side as its parent, and that quantities are conserved down the chain.

In [ ]:
for name, spec in SCHEMAS.items():
    print(f"{name:14s}", ', '.join(c + ('' if req else '?') for c, (_, req) in spec.items()))

## 2. Market data per stock-day and per venue

`bmll` mode needs a point-in-time PERMNO → ticker map. It is built from the CRSP tickers already in `weights.csv`. Check any names BMLL does not recognise, such as share-class tickers like BRK.B.

In [ ]:
from tca.market_data import synthetic_market, anchors_from_blotter, ticker_map_from_weights, BMLLSource
event_days = pm.trade_date.unique()
if DATA_SOURCE == 'synthetic':
    venue_days, stock_days = synthetic_market(anchors_from_blotter(blotter, weights),
                                              seed=SEED, index_event_dates=event_days)
elif DATA_SOURCE == 'bmll':
    import bmll
    src = BMLLSource(bmll)
    venue_days, stock_days = src.market(pm.rename(columns={'trade_date': 'date'})[['date', 'permno']],
                                        ticker_map_from_weights(weights), index_event_dates=event_days)
    print('Resolved BMLL metrics:', src.resolved)
else:
    raise ValueError(DATA_SOURCE)
venue_days.to_csv(OUT/f'venue_days_{DATA_SOURCE}.csv', index=False)
stock_days.to_csv(OUT/f'stock_days_{DATA_SOURCE}.csv', index=False)
share = venue_days.groupby('mic').volume.sum().sort_values(ascending=False)
print(f'{venue_days.mic.nunique()} venues; continuous volume share by MIC:')
display((share/share.sum()).round(3).to_frame('share').T)

## 3. One rebalance date: three executions of the same orders

* **CLOSE_AUCTION**: MOC on the primary listing, capped at 25% of the *expected* auction (ADV × historical auction share). Any excess is worked as VWAP.
* **VWAP**: 09:30–15:50 along the intraday volume curve, split across MICs by venue volume, capped at 10% of expected continuous volume. Any excess goes to the close.
* **ARRIVAL_IS**: front-loaded from the open, priced at **arrival + half-spread + square-root impact**. Any excess goes to the close.

In [ ]:
from tca.analysis import run_strategies, weighted, COST_COLS, rebalance_costs, execution_tracking
from tca.execution import ImpactModel
model = ImpactModel(**IMPACT)
by_date = pm.groupby('trade_date').notional_usd.sum()
focus = pd.Timestamp(FOCUS_TRADE_DATE) if FOCUS_TRADE_DATE else by_date.idxmax()
one = pm[pm.trade_date.eq(focus)]
print(f"Focus rebalance: trade date {focus.date()} ({focus.day_name()}), {len(one)} orders, "
      f"${one.notional_usd.sum()/1e6:,.0f}M")
runs = run_strategies(one, venue_days, stock_days, model=model, **GEN)
for k, r in runs.items():
    validate_links(one, r['trader'], r['broker'], r['fills'].drop(columns=['trader_order_id', 'permno']))
    print(f"{k:14s} trader orders {len(r['trader']):5d} | child orders {len(r['broker']):6d} | "
          f"fills {len(r['fills']):6d} | MICs {r['broker'].mic.nunique()}")
focus_orders = pd.concat([r['orders'] for r in runs.values()])
summary = weighted(focus_orders, COST_COLS, 'strategy')
display(summary[COST_COLS + ['notional_usd', 'cost_vs_close_usd']].style.format(
    {**{c: '{:.2f}' for c in COST_COLS}, 'notional_usd': '${:,.0f}', 'cost_vs_close_usd': '${:,.0f}'}))
focus_orders.to_csv(OUT/f'focus_{focus.date()}_order_tca.csv', index=False)

### Example: one order down the PM → Trader → Broker → Fill chain (VWAP)

In [ ]:
big = one.nlargest(1, 'notional_usd').iloc[0]
r = runs['VWAP']
t = r['trader'][r['trader'].pm_order_id.eq(big.pm_order_id)]
b = r['broker'][r['broker'].trader_order_id.isin(t.trader_order_id)]
f = r['fills'][r['fills'].child_order_id.isin(b.child_order_id)]
display(pd.DataFrame([big]))
display(t)
display(b.groupby(['mic', 'order_type']).quantity.sum().sort_values(ascending=False).to_frame().T)
print(f"{len(b)} children across {b.mic.nunique()} MICs and {b.bucket_start.nunique()} buckets; "
      f"filled {f.quantity.sum():,} of {big.quantity:,} shares at avg {np.average(f.price, weights=f.quantity):.4f}")

### Where the cost comes from

Cost is concentrated in the few large orders, mostly index adds and deletes. Bucketing by order size as % of ADV shows each strategy's trade-off: the auction pays impact only, while VWAP and arrival also carry the **intraday price move away from the close**, which is risk relative to the index rather than expected cost.

In [ ]:
fo = focus_orders.copy()
fo['size_bucket'] = pd.cut(fo.pct_adv, [0, 1e-4, 1e-3, 1e-2, 3e-2, 10],
                           labels=['<0.01%', '0.01-0.1%', '0.1-1%', '1-3%', '>3%'])
tab = fo.groupby(['size_bucket', 'strategy'], observed=True).apply(
    lambda g: pd.Series({'orders': len(g), 'notional_share': g.notional_usd.sum(),
                         'vs_close_bps': np.average(g.vs_close_bps, weights=g.notional_usd),
                         'abs_vs_close_bps': np.average(g.vs_close_bps.abs(), weights=g.notional_usd)}),
    include_groups=False).unstack('strategy')
tab['notional_share'] = tab['notional_share'] / fo.groupby('strategy').notional_usd.sum().iloc[0]
display(tab.round(2))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for k, g in fo.groupby('strategy'):
    ax[0].hist(g.vs_close_bps.clip(-150, 150), bins=60, histtype='step', label=k, density=True)
ax[0].set_xlabel('cost vs close (bps, clipped)'); ax[0].legend(); ax[0].set_title(f'Order-level cost, {focus.date()}')
mic = runs['VWAP']['fills'].groupby('mic').apply(lambda g: (g.quantity * g.price).sum(), include_groups=False)
(mic / mic.sum()).sort_values().plot.barh(ax=ax[1], color='#2a5c8a')
ax[1].set_title('VWAP notional by MIC'); ax[1].set_xlabel('share of executed notional')
fig.tight_layout(); plt.show()

## 4. All rebalances: notional-weighted costs and execution tracking error

Each rebalance's cost vs close, in bps of the decision-day NAV, is an active return on that day. Over the evaluation window:
* **Tracking difference** = −Σ cost / years.
* **Tracking error** = √(Σ cost² / years). This equals daily TE × √252 when other days contribute nothing.
* **TE dispersion** = std(cost) × √(rebalances per year), i.e. only the unpredictable part.

In [ ]:
all_orders = []
for d, g in pm.groupby('trade_date'):
    rr = run_strategies(g, venue_days, stock_days, model=model, **GEN)
    all_orders += [x['orders'] for x in rr.values()]
all_orders = pd.concat(all_orders, ignore_index=True)
reb = rebalance_costs(all_orders, nav)
years = (pd.Timestamp(END_DATE) - pd.Timestamp(ANALYSIS_START)).days / 365.25
in_window = reb[reb.trade_date.between(ANALYSIS_START, END_DATE)]
te = execution_tracking(in_window, years)
display(te.style.format('{:.2f}', subset=[c for c in te.columns if c != 'rebalances']))
overall = weighted(all_orders, COST_COLS, 'strategy')
display(overall[COST_COLS].round(2))
all_orders.to_csv(OUT/f'all_rebalances_order_tca_{DATA_SOURCE}.csv', index=False)
reb.to_csv(OUT/f'rebalance_costs_{DATA_SOURCE}.csv', index=False)
te.to_csv(OUT/f'execution_tracking_{DATA_SOURCE}.csv')
_ = (OUT/f'assumptions_{DATA_SOURCE}.json').write_text(json.dumps(
    {'data_source': DATA_SOURCE, 'impact': model.as_dict(), 'generator': GEN, 'seed': SEED,
     'blotter': str(BLOTTER_CSV if not ORDERS_CSV else ORDERS_CSV), 'reasons': REASONS}, indent=2))

In [ ]:
piv = reb.pivot(index='trade_date', columns='strategy', values='cost_vs_close_bps_nav')
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
piv.plot(ax=ax[0], marker='o'); ax[0].axhline(0, color='grey', lw=.6)
ax[0].set_ylabel('cost vs close (bps of NAV)'); ax[0].set_title('Per-rebalance execution cost')
(-piv.cumsum()).plot(ax=ax[1]); ax[1].set_ylabel('cumulative active return (bps)')
ax[1].set_title('Cumulative tracking difference from execution'); fig.tight_layout(); plt.show()
display(reb.pivot(index='trade_date', columns='strategy', values='turnover_pct_nav').iloc[:, :1].round(2)
        .rename(columns=lambda c: 'turnover_pct_nav'))

## 5. Reading the results and what remains

* For an index fund, the **closing auction** is the reference execution because the benchmark itself trades there. Its cost is pure impact plus fees, and its tracking error is small. **VWAP** and **arrival** strategies trade before the close, so they add market risk against the benchmark. Part of that risk nets out across each rebalance's two-sided buys and sells, and the rest shows up as tracking error.
* With `DATA_SOURCE='synthetic'`, the intraday path and liquidity are simulated, so **only the relative mechanics are meaningful**. Re-run with `DATA_SOURCE='bmll'` before drawing conclusions.
* **Calibrate** `eta` and `auction_eta` against BMLL markouts and auction imbalance data. The defaults are literature values.
* **Re-point the blotter** at the Friday-auction re-run of `BMLL_4Y_Corrected_Portfolio_Orderbook.ipynb`. The handoff file still trades on Tuesday, after the index has already rebalanced.
* Possible extensions: intraday per-venue curves from BMLL trades instead of the default U-shape; auction imbalance feeds; mixed strategies (e.g. 70% MOC + 30% VWAP) as extra trader legs.